In [0]:
%run /Workspace/weather_notebook/nb_utils_dev

In [0]:
# ── Cell 2: Run Silver notebooks ─────────────────────────────
print("="*60)
print("  SILVER ORCHESTRATOR — UK WEATHER")
print(f"  Catalog: {catalog}")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

base = "/Workspace/weather_notebook"

notebooks = [
    ("weather_readings",     f"{base}/nb_silver_weather_dev"),
    ("air_quality_readings", f"{base}/nb_silver_airquality_dev"),
]

results = {}
failed  = []

for table, path in notebooks:
    print(f"\nRunning: {path.split('/')[-1]}")
    try:
        result = dbutils.notebook.run(path, timeout_seconds=1800)
        parts  = result.split("|")
        results[parts[0]] = {"rows": int(parts[1]), "status": parts[2]}
        print(f"  Result: {parts[0]} | {int(parts[1]):,} rows | {parts[2]}")
    except Exception as e:
        results[table] = {"rows": 0, "status": "FAILED"}
        failed.append(table)
        print(f"  FAILED: {table} — {str(e)}")

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*60)
print("  SILVER ORCHESTRATOR — SUMMARY")
print("="*60)
for table, result in results.items():
    icon = "✓" if result["status"] == "PASS" else "✗"
    print(f"  {icon} {table:25} | {result['rows']:>6,} rows | {result['status']}")

print(f"\n  Catalog: {catalog}")
print(f"  Tables:  {silver_catalog}.weather_readings")
print(f"           {silver_catalog}.air_quality_readings")
print(f"  Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

if failed:
    raise Exception(f"Silver failed for: {', '.join(failed)}")

dbutils.notebook.exit("SUCCESS")